# UC Irvine Online Retailer - Demand Forecasting Recipe for KumoRFM

[[**Blog**](https://kumo.ai/company/news/kumo-relational-foundation-model/) | [**Paper**](https://kumo.ai/research/kumo_relational_foundation_model.pdf)]

**KumoRFM (Kumo Relational Foundation Model)** is a Foundation Model for machine learning on enterprise data. With just your data and a few lines of code, you can generate accurate predictions in real-time: no model training or pipelines required.

This notebook provides a **recipe for demand forecasting/predicting item sales amount** on the [UC Irvine Online Retailer](https://archive.ics.uci.edu/dataset/352/online+retail) dataset.

In [ ]:
!pip install kumoai --pre

## Initialize KumoRFM

`KumoRFM` provides an [SDK](https://kumo-ai.github.io/kumo-sdk/docs/get_started/rfm/index.html) in Python.
The Kumo SDK is available for Python 3.10 to Python 3.14.

In [ ]:
from kumoai.experimental import rfm

You will need an API key to make calls to KumoRFM.

In [ ]:
rfm.authenticate()

In [ ]:

rfm.init()

## Dataset Explanation
This project uses publicly available S3 datasets related to UCI Online Retail transactions, structured as **normalized tables**. The base S3 path for these datasets is `s3://kumo-public-datasets/uci_online_retailer/`. Here's a quick overview of each resource:

*   **`dim_country.parquet`**: Country dimension (`country_id`, `country_name`).
*   **`dim_customer.parquet`**: Customer dimension with `customer_id`, `country_id`, and `guest` flag.
*   **`dim_item.parquet`**: Item dimension (`item_id`, `stock_code`, `description`) – **This is the main entity table used by RFM.**
*   **`fact_transaction_header.parquet`**: Invoice-level transaction header (`txn_id`, `invoice_datetime`, `customer`, `country`).
*   **`fact_transaction_line.parquet`**: Line-level transaction facts (`item_id`, `quantity`, `unit_price`, `line_amount`).
*   **`item_daily_active.parquet`**: Day-level per-item aggregates (`daily_amount`, `daily_quantity`, `txn_line_count`, `info_available_date`) – **This is the time-series signal table.**

We load all the necessary data into memory using `pandas`.

In [ ]:
import pandas as pd

item_df = pd.read_parquet("s3://kumo-public-datasets/uci_online_retailer/dim_item.parquet")
country_df = pd.read_parquet("s3://kumo-public-datasets/uci_online_retailer/dim_country.parquet")
customer_df = pd.read_parquet("s3://kumo-public-datasets/uci_online_retailer/dim_customer.parquet")
txn_header_df = pd.read_parquet("s3://kumo-public-datasets/uci_online_retailer/fact_transaction_header.parquet")
txn_line_df = pd.read_parquet("s3://kumo-public-datasets/uci_online_retailer/fact_transaction_line.parquet")
daily_active_df = pd.read_parquet("s3://kumo-public-datasets/uci_online_retailer/item_daily_active.parquet")
daily_active_df["info_available_date"] = pd.to_datetime(daily_active_df["info_available_date"]).dt.normalize()

For regression tasks, it can be beneficial to explore various target encodings (*e.g.* in case of huge outliers or variance). Here, our agent identified a better target value encoding, which will be a part of the product soon.

In particular, we scale each item's daily amount by its historical mean:

$$
y_{i,\hat{t}} = \frac{y_{i,\hat{t}}}{\sqrt{\max(\frac{1}{N_i} \sum_{t<T}~~y_{i,t} - 16, 0)}+1}.
$$

This normalizes each item's values by dividing by a scale that increases with its historical mean (only above a threshold), so high-volume items are dampened while low-volume ones remain unchanged, making the series more comparable and stable for modeling.

In [ ]:
anchor_time = pd.Timestamp("2011-11-19")

scale_df = (
    daily_active_df[daily_active_df["date"] < anchor_time]
    .groupby("item_id", as_index=False)["daily_amount"]
    .mean().rename(columns={"daily_amount": "mean_label"})
)
scale_df["scale_factor"] = (scale_df["mean_label"] - 16).clip(lower=0).pow(0.5) + 1.0
scale_df = scale_df[["item_id", "scale_factor"]]

daily_active_df = daily_active_df.merge(scale_df, on="item_id", how="left").fillna(1.0)
daily_active_df["daily_amount_scaled"] = daily_active_df["daily_amount"] / daily_active_df["scale_factor"]

Using the loaded raw table data, we register them as a `LocalTable` in `KumoRFM` and connect them via primary key<>foreign key relationship.

In [ ]:
item_tbl = rfm.LocalTable(
    item_df,
    name="item",
    primary_key="item_id",
)
daily_tbl = rfm.LocalTable(
    daily_active_df,
    name="daily_item",
    time_column="info_available_date",
)
country_tbl = rfm.LocalTable(
    country_df,
    name="country",
    primary_key="country_id",
)
customer_tbl = rfm.LocalTable(
    customer_df,
    name="customer",
    primary_key="customer_id",
)
txh_tbl = rfm.LocalTable(
    txn_header_df,
    name="txn_header",
    primary_key="txn_id",
    time_column="invoice_datetime",
)
txl_tbl = rfm.LocalTable(
    txn_line_df,
    name="txn_line",
    primary_key="txn_line_id",
)

graph = rfm.Graph(tables=[item_tbl, daily_tbl, country_tbl, customer_tbl, txh_tbl, txl_tbl])
graph.link(src_table="daily_item", fkey="item_id", dst_table="item")
graph.link(src_table="customer", fkey="country_id", dst_table="country")
graph.link(src_table="txn_header", fkey="country_id", dst_table="country")
graph.link(src_table="txn_header", fkey="customer_id", dst_table="customer")
graph.link(src_table="txn_line", fkey="item_id", dst_table="item")
graph.link(src_table="txn_line", fkey="txn_id", dst_table="txn_header")

graph.print_metadata()
graph.print_links()

In [ ]:
graph.visualize()

## Query KumoRFM

Now that the graph is setup, we make predictions for next 7 day amount sales using KumoRFM:

- Prediction date: `2011-11-18`
- Forecast horizon: next 7 days
- Prediction entity: `item.item_id`

In [ ]:
model = rfm.KumoRFM(graph)

In [ ]:
query = """
PREDICT SUM(daily_item.daily_amount_scaled, 0, 7, days)
FOR EACH item.item_id
"""
with model.batch_mode():
    df = model.predict(
        query,
        indices=item_df["item_id"].astype(int).tolist(),
        anchor_time=anchor_time,
        run_mode="best",
    )

As we encoded the target value before, we decode the predicted values back by multiplying the scale for each entity:

In [ ]:
pred_df = df.rename(columns={"ENTITY": "item_id", "TARGET_PRED": "y_pred_scaled"})
pred_df = pred_df.merge(scale_df[["item_id", "scale_factor"]], on="item_id", how="left").fillna({"scale_factor": 1.0})
pred_df["y_pred"] = pred_df["y_pred_scaled"] * pred_df["scale_factor"]
pred_df.head()

## Evaluation

We can now gather the actual ground-truth targets from the data to evaluate against:

In [ ]:
from datetime import timedelta

start_date = anchor_time + timedelta(days=0)
end_date = anchor_time + timedelta(days=6)

# Filter daily_active_df for the 7-day period
filtered_daily_df = daily_active_df[(daily_active_df['date'] >= start_date) & (daily_active_df['date'] <= end_date)]

# Group by item_id and sum daily_amount
ground_truth = filtered_daily_df.groupby('item_id')['daily_amount'].sum().reset_index()
print(ground_truth.head())

We evaluate using Mean Absolute Error (`MAE`), R-Squared (`R2`), and Root Mean Squared Error (`RMSE`):

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
import numpy as np

ground_truth = ground_truth.rename(columns={'daily_amount': 'y_true'})
pred_df = pred_df.merge(ground_truth, on='item_id', how='right').fillna(0.0)
y_true = pred_df['y_true'].to_numpy(dtype=float)
y_pred = pred_df['y_pred'].to_numpy(dtype=float)

metric_mae = mean_absolute_error(y_true, y_pred)
metric_r2 = r2_score(y_true, y_pred)
metric_rmse = np.sqrt(mean_squared_error(y_true, y_pred))

print(f"MAE: {metric_mae}")
print(f"R2: {metric_r2}")
print(f"RMSE: {metric_rmse}")


<div align="left">
  <img src="https://kumo-sdk-public.s3.us-west-2.amazonaws.com/rfm-colabs/kumo_ai_logo.jpeg" width="30" />
</div>